# YuE2 Studio 5.0.0 — Kaggle Studio (Cloudflare)

**This launches the actual upstream YuE2 Studio 5.0.0 backend and its real React interface. It is not ComfyUI, Gradio, or a clone.**

## Before running
1. Run both code cells in the **YuE2 Studio 5.0.0 — Assets Builder** notebook and save the output, or publish the folder as a private Kaggle Dataset.
2. Attach that builder output here using Add Input → Notebook Output Files (or Add Input → Your Datasets).
3. Set Kaggle Internet ON and Accelerator GPU T4 ×2.
4. Run the launch cell. It verifies the assets, installs the Linux runtime libraries, starts the official Studio 5.0.0 AppImage in upstream headless mode under a virtual display, checks its API, serves the prebuilt upstream browser interface, and opens a Cloudflare Quick Tunnel.
5. Open the printed URL and enter the generated password.

## What is running
- Backend: the official YuE2 Studio 5.0.0 Linux AppImage, started with upstream --headless. Its local API listens on port 8791.
- Interface: the real React application built from the upstream v5.0.0 source tag. It is served on port 3791. API and MCP requests are reverse-proxied to the Studio backend on the same origin, including streaming responses.
- Remote access: cloudflared exposes only the browser interface. HTTP Basic Authentication protects both the website and proxied API. The backend port 8791 is not exposed directly.

The Studio downloads selected engine/model components through its own Models/Setup screens. Start with YuE2, then download ACE-Step or MiniMax components inside the Studio as needed. The Studio data, model downloads and generated songs live under /kaggle/working/.local/share/yue2-studio.

## Browser limitations
This is the actual upstream web frontend, but the upstream Tauri desktop shell is headless inside the Kaggle container. Features that require native Tauri windows, a system tray, OS notifications, Windows-only audio capture, or native file dialogs are not available in the remote browser. Use the Studio's in-page import/export controls. The Linux release is marked experimental upstream; the Kaggle Linux/container/GPU combination must still be tested in a live session.

The Cloudflare URL and password are temporary and only work while the Kaggle session remains alive. Keep the password private. Save notebook output to preserve generated songs and downloaded models between sessions.

In [ ]:
import os
import json
import re
import time
import stat
import shutil
import secrets
import hashlib
import subprocess
from pathlib import Path

import requests

STUDIO_VERSION = "5.0.0"
STUDIO_TAG = "v5.0.0"
MANIFEST_NAME = "yue2-studio-assets.json"
WORK = Path("/kaggle/working/yue2_studio_runtime")
WORK.mkdir(parents=True, exist_ok=True)
LOGS = WORK / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

def run(command, check=True, **kwargs):
    return subprocess.run(command, check=check, text=True, **kwargs)

def tail(path, lines=70):
    path = Path(path)
    if not path.exists():
        return "(log not created)"
    return "\n".join(path.read_text(errors="replace").splitlines()[-lines:])

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# Stop processes from a previous run in this same notebook kernel.
for old_name in ("CLOUDFLARED_PROCESS", "BROWSER_PROCESS", "STUDIO_PROCESS"):
    old = globals().get(old_name)
    if old is not None and old.poll() is None:
        old.terminate()
        try:
            old.wait(timeout=12)
        except Exception:
            old.kill()
            old.wait(timeout=5)

# Locate the builder output whether it is attached as notebook output or as a private Kaggle Dataset.
manifest_candidates = []
for root in (Path("/kaggle/input"), Path("/kaggle/working")):
    if root.exists():
        manifest_candidates.extend(root.rglob(MANIFEST_NAME))
if not manifest_candidates:
    raise FileNotFoundError(
        "Cannot find yue2-studio-assets.json. Attach the output of "
        "YuE2_Studio_5_0_0_Assets_Builder.ipynb under Add Input."
    )
manifest_path = manifest_candidates[0]
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest.get("studio_version") != STUDIO_VERSION or manifest.get("studio_tag") != STUDIO_TAG:
    raise RuntimeError(f"Expected YuE2 Studio {STUDIO_TAG}; assets manifest says {manifest.get('studio_tag')}.")
if not manifest.get("frontend", {}).get("source_commit"):
    raise RuntimeError("The attached assets do not include the compiled Studio frontend. Run both builder code cells again.")

ASSET_DIR = manifest_path.parent
APPIMAGE_SOURCE = ASSET_DIR / manifest["appimage"]["name"]
CLOUDFLARED_SOURCE = ASSET_DIR / manifest["cloudflared"]["name"]
WEB_SOURCE = ASSET_DIR / "studio-web"
NODE_SOURCE = ASSET_DIR / "node-runtime" / "node"
for required in (APPIMAGE_SOURCE, CLOUDFLARED_SOURCE, WEB_SOURCE / "index.html", WEB_SOURCE / "visualizer.html", NODE_SOURCE):
    if not required.exists():
        raise FileNotFoundError(f"Missing required Studio asset: {required}")

# Validate the official binaries and every file in the built website.
for path, item in (
    (APPIMAGE_SOURCE, manifest["appimage"]),
    (CLOUDFLARED_SOURCE, manifest["cloudflared"]),
    (NODE_SOURCE, manifest["node_runtime"]),
):
    digest = sha256_file(path)
    if digest.lower() != item["sha256"].lower():
        raise RuntimeError(f"SHA-256 mismatch for {path.name}: {digest}")
for relative, item in manifest["frontend"]["files"].items():
    file_path = WEB_SOURCE / relative
    if not file_path.is_file() or file_path.stat().st_size != item["size"] or sha256_file(file_path) != item["sha256"]:
        raise RuntimeError(f"Frontend asset failed integrity validation: {relative}")

APPIMAGE = WORK / manifest["appimage"]["name"]
CLOUDFLARED = WORK / "cloudflared-linux-amd64"
NODE = WORK / "node"
WEB = WORK / "studio-web"
shutil.copy2(APPIMAGE_SOURCE, APPIMAGE)
shutil.copy2(CLOUDFLARED_SOURCE, CLOUDFLARED)
shutil.copy2(NODE_SOURCE, NODE)
if WEB.exists():
    shutil.rmtree(WEB)
shutil.copytree(WEB_SOURCE, WEB)
for path in (APPIMAGE, CLOUDFLARED, NODE):
    path.chmod(path.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

node_version = subprocess.check_output([str(NODE), "--version"], text=True).strip()
if node_version != manifest["node_runtime"]["version"]:
    raise RuntimeError(f"Unexpected Node version: {node_version}")
print(f"Verified official YuE2 Studio {STUDIO_VERSION}, upstream UI commit {manifest['frontend']['source_commit'][:12]}, cloudflared and Node {node_version}.")

# Tauri/WebKit dependencies plus a virtual display for the official upstream headless Linux application.
print("Installing Linux runtime dependencies...")
run(["bash", "-lc",
     "apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends "
     "xvfb xauth dbus-x11 libwebkit2gtk-4.1-0 libayatana-appindicator3-1 "
     "libgtk-3-0 librsvg2-2 libasound2 libgomp1 libvulkan1 ca-certificates "
     "gstreamer1.0-plugins-base gstreamer1.0-plugins-good gstreamer1.0-plugins-bad "
     "gstreamer1.0-libav >/tmp/yue2-studio-apt.log 2>&1 || "
     "{ tail -80 /tmp/yue2-studio-apt.log; exit 1; }"])

# Kaggle Output preserves the Studio library, downloaded weights, settings and songs between saved versions.
STATE = Path("/kaggle/working/.local")
XDG_DATA = STATE / "share"
XDG_CONFIG = STATE / "config"
XDG_CACHE = STATE / "cache"
XDG_RUNTIME = WORK / "xdg-runtime"
for folder in (XDG_DATA, XDG_CONFIG, XDG_CACHE, XDG_RUNTIME):
    folder.mkdir(parents=True, exist_ok=True)
XDG_RUNTIME.chmod(0o700)

studio_env = os.environ.copy()
studio_env.update({
    "APPIMAGE_EXTRACT_AND_RUN": "1",
    "GDK_BACKEND": "x11",
    "WEBKIT_DISABLE_COMPOSITING_MODE": "1",
    "WEBKIT_DISABLE_DMABUF_RENDERER": "1",
    "NO_AT_BRIDGE": "1",
    "XDG_DATA_HOME": str(XDG_DATA),
    "XDG_CONFIG_HOME": str(XDG_CONFIG),
    "XDG_CACHE_HOME": str(XDG_CACHE),
    "XDG_RUNTIME_DIR": str(XDG_RUNTIME),
    "YUE_STUDIO_IDLE_SHUTDOWN_S": "86400",
})

studio_log = LOGS / "studio-headless.log"
print("Starting the official YuE2 Studio backend in upstream headless mode...")
with open(studio_log, "w", encoding="utf-8") as log_file:
    STUDIO_PROCESS = subprocess.Popen(
        [
            "dbus-run-session", "--",
            "xvfb-run", "-a", "-s", "-screen 0 1600x1000x24",
            str(APPIMAGE), "--headless",
        ],
        cwd=WORK,
        env=studio_env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

api_ready = False
for attempt in range(90):
    if STUDIO_PROCESS.poll() is not None:
        raise RuntimeError(
            f"YuE2 Studio exited during startup (code {STUDIO_PROCESS.returncode}).\n"
            + tail(studio_log)
        )
    try:
        response = requests.get("http://127.0.0.1:8791/health", timeout=4)
        if response.status_code < 500:
            api_ready = True
            print("Studio API ready:", response.status_code)
            break
    except requests.RequestException:
        pass
    time.sleep(2)
if not api_ready:
    raise RuntimeError("Studio API did not become ready on port 8791.\n" + tail(studio_log))

# This local Node server serves the compiled upstream React UI, proxies all official API/MCP paths,
# and requires HTTP Basic Auth. Streamed responses (including event streams) are piped through.
NODE_SERVER = WORK / "studio-browser-server.mjs"
NODE_SERVER.write_text(r'''
import http from 'node:http';
import fs from 'node:fs';
import path from 'node:path';
import { timingSafeEqual } from 'node:crypto';

const root = path.resolve(process.env.JJP_STUDIO_WEB || '');
const password = process.env.JJP_STUDIO_GATE_PASSWORD || '';
const port = Number(process.env.JJP_STUDIO_PORT || '3791');
const apiHost = '127.0.0.1';
const apiPort = 8791;
const apiPrefixes = ['/v1', '/setup', '/engine', '/health', '/mcp'];
const mime = {
  '.html':'text/html; charset=utf-8', '.js':'text/javascript; charset=utf-8',
  '.mjs':'text/javascript; charset=utf-8', '.css':'text/css; charset=utf-8',
  '.json':'application/json; charset=utf-8', '.svg':'image/svg+xml',
  '.png':'image/png', '.jpg':'image/jpeg', '.jpeg':'image/jpeg',
  '.webp':'image/webp', '.ico':'image/x-icon', '.woff':'font/woff',
  '.woff2':'font/woff2', '.ttf':'font/ttf', '.mp3':'audio/mpeg',
  '.wav':'audio/wav', '.mp4':'video/mp4', '.wasm':'application/wasm',
  '.txt':'text/plain; charset=utf-8'
};
function authorised(header) {
  const expected = 'Basic ' + Buffer.from('studio:' + password).toString('base64');
  if (typeof header !== 'string') return false;
  const a = Buffer.from(header);
  const b = Buffer.from(expected);
  return a.length === b.length && timingSafeEqual(a, b);
}
function apiPath(urlPath) {
  return apiPrefixes.some(p => urlPath === p || urlPath.startsWith(p + '/'));
}
function sendError(res, status, message) {
  if (res.headersSent) return res.destroy();
  res.writeHead(status, {'Content-Type':'text/plain; charset=utf-8','Cache-Control':'no-store'});
  res.end(message);
}
const server = http.createServer((req, res) => {
  if (!authorised(req.headers.authorization)) {
    res.writeHead(401, {
      'WWW-Authenticate':'Basic realm="JesusJuiceProductions YuE2 Studio"',
      'Content-Type':'text/plain; charset=utf-8',
      'Cache-Control':'no-store'
    });
    res.end('Studio sign-in required');
    return;
  }
  const url = new URL(req.url || '/', 'http://studio.local');
  if (apiPath(url.pathname)) {
    const headers = {...req.headers, host:apiHost + ':' + apiPort};
    delete headers.authorization;
    delete headers['proxy-authorization'];
    delete headers.connection;
    const upstream = http.request({
      hostname:apiHost, port:apiPort, method:req.method || 'GET',
      path:(req.url || '/'), headers
    }, upstreamRes => {
      const responseHeaders = {...upstreamRes.headers};
      delete responseHeaders.connection;
      delete responseHeaders['keep-alive'];
      delete responseHeaders['proxy-authenticate'];
      delete responseHeaders['proxy-authorization'];
      delete responseHeaders['transfer-encoding'];
      res.writeHead(upstreamRes.statusCode || 502, responseHeaders);
      upstreamRes.pipe(res);
    });
    upstream.on('error', error => sendError(res, 502, 'Studio API proxy error: ' + error.message));
    req.on('aborted', () => upstream.destroy());
    req.pipe(upstream);
    return;
  }
  let pathname;
  try { pathname = decodeURIComponent(url.pathname); }
  catch { return sendError(res, 400, 'Bad URL'); }
  let file = path.resolve(root, '.' + pathname);
  if (file !== root && !file.startsWith(root + path.sep)) return sendError(res, 403, 'Forbidden');
  try {
    if (fs.existsSync(file) && fs.statSync(file).isDirectory()) file = path.join(file, 'index.html');
    if (!fs.existsSync(file) || !fs.statSync(file).isFile()) {
      if (path.extname(pathname)) return sendError(res, 404, 'Not found');
      file = path.join(root, 'index.html');
    }
    const info = fs.statSync(file);
    const headers = {
      'Content-Type':mime[path.extname(file).toLowerCase()] || 'application/octet-stream',
      'Content-Length':info.size,
      'Accept-Ranges':'bytes',
      'Cache-Control':path.basename(file) === 'index.html' ? 'no-store' : 'public, max-age=3600'
    };
    if (req.method === 'HEAD') {
      res.writeHead(200, headers);
      return res.end();
    }
    if (req.method !== 'GET') return sendError(res, 405, 'Method not allowed');
    const range = req.headers.range;
    if (range) {
      const match = /^bytes=(\d*)-(\d*)$/.exec(range);
      if (match) {
        let start = match[1] ? Number(match[1]) : 0;
        let end = match[2] ? Number(match[2]) : info.size - 1;
        if (!match[1] && match[2]) {
          start = Math.max(0, info.size - Number(match[2]));
          end = info.size - 1;
        }
        if (Number.isSafeInteger(start) && Number.isSafeInteger(end) && start <= end && end < info.size) {
          headers['Content-Range'] = 'bytes ' + start + '-' + end + '/' + info.size;
          headers['Content-Length'] = end - start + 1;
          res.writeHead(206, headers);
          return fs.createReadStream(file, {start,end}).pipe(res);
        }
      }
      return sendError(res, 416, 'Range not satisfiable');
    }
    res.writeHead(200, headers);
    fs.createReadStream(file).pipe(res);
  } catch (error) {
    sendError(res, 500, 'Studio web server error: ' + error.message);
  }
});
server.listen(port, '0.0.0.0', () => {
  console.log('YuE2 Studio web UI listening on port ' + port);
});
''', encoding="utf-8")

gate_password = secrets.token_urlsafe(18)
browser_env = studio_env.copy()
browser_env.update({
    "JJP_STUDIO_WEB": str(WEB),
    "JJP_STUDIO_GATE_PASSWORD": gate_password,
    "JJP_STUDIO_PORT": "3791",
})
browser_log = LOGS / "studio-browser.log"
print("Serving the upstream YuE2 Studio interface...")
with open(browser_log, "w", encoding="utf-8") as log_file:
    BROWSER_PROCESS = subprocess.Popen(
        [str(NODE), str(NODE_SERVER)],
        cwd=WORK,
        env=browser_env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

ui_ready = False
for attempt in range(45):
    if BROWSER_PROCESS.poll() is not None:
        raise RuntimeError(
            f"Studio browser server exited (code {BROWSER_PROCESS.returncode}).\n"
            + tail(browser_log)
        )
    try:
        response = requests.get("http://127.0.0.1:3791/", auth=("studio", gate_password), timeout=5)
        if response.status_code == 200 and "YuE2" in response.text:
            ui_ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)
if not ui_ready:
    raise RuntimeError("Studio browser UI did not become ready on port 3791.\n" + tail(browser_log))

proxy_check = requests.get(
    "http://127.0.0.1:3791/health",
    auth=("studio", gate_password),
    timeout=10,
)
if proxy_check.status_code >= 500:
    raise RuntimeError(f"Studio API proxy check failed: {proxy_check.status_code} {proxy_check.text[:500]}")

# Only the authenticated browser server is exposed. Port 8791 remains private on localhost.
cloudflared_log = LOGS / "cloudflared.log"
with open(cloudflared_log, "w", encoding="utf-8") as log_file:
    CLOUDFLARED_PROCESS = subprocess.Popen(
        [
            str(CLOUDFLARED), "tunnel",
            "--no-autoupdate",
            "--loglevel", "info",
            "--url", "http://127.0.0.1:3791",
        ],
        cwd=WORK,
        env=studio_env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

public_url = None
for attempt in range(90):
    if CLOUDFLARED_PROCESS.poll() is not None:
        raise RuntimeError(
            f"cloudflared exited (code {CLOUDFLARED_PROCESS.returncode}).\n"
            + tail(cloudflared_log)
        )
    log_text = tail(cloudflared_log, 180)
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text, re.I)
    if match:
        public_url = match.group(0)
        break
    time.sleep(2)
if not public_url:
    raise RuntimeError("Cloudflare Quick Tunnel did not provide a public URL.\n" + tail(cloudflared_log))

try:
    edge_check = requests.get(public_url, auth=("studio", gate_password), timeout=25)
    print("Cloudflare edge check:", edge_check.status_code)
except requests.RequestException as exc:
    print("Cloudflare tunnel issued a URL; first browser open may complete edge warm-up:", str(exc)[:240])

print("\n" + "=" * 72)
print("YUЕ2 STUDIO 5.0.0 IS RUNNING")
print("Studio URL :", public_url)
print("Username   : studio")
print("Password   :", gate_password)
print("Studio data:", XDG_DATA / "yue2-studio")
print("Logs       :", LOGS)
print("=" * 72)
print("Keep this Kaggle session alive while using the Studio. Save Kaggle output to preserve models and songs.")